In [16]:
import re
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)

df = pd.read_csv('cleaning_data.csv')

KEY_COLS  = ['Order_ID', 'Customer_ID']
DATE_COLS = ['Order_Date', 'Ship_Date']

print(f'Shape: {df.shape[0]:,} rows × {df.shape[1]} columns')
df.head()

Shape: 100 rows × 18 columns


,Order_ID,Customer_ID,Customer_Name,Email,Country,City,Order_Date,Ship_Date,Product_Category,Quantity,Unit_Price,Discount,Sales,Age,Customer_Segment,Satisfaction_Score,Payment_Method,Notes
0,O1001,C016,MAHA ZADEH,maha.zadeh@example.com,Deutschland,Mannheim,2026/01/27,not available,office supplies,2,19.73,0.25,29.59,60.0,Corporate,5.0,Bank Transfer,ok
1,O1002,C025,Sara Smith,sara.smith@example.com,germany,karlsruhe,03-30-2026,2026-04-08,electronics,7,285.86,0.10,1800.92,52.0,CORPORATE,NaN,Bank Transfer,ok
2,O1003,C014,Carlos Garcia,carlos.garcia@example.com,Italy,Rome,19/01/2026,2026/01/28,Office Supplies,7,312.17,0.10,1966.67,69.0,consumer,1.0,NaN,check address
3,O1004,C009,Nina Weber,nina.weber@example.com,Deutschland,Mannheim,2026/05/18,22.05.2026,Home,4,226.13,0.00,904.52,25.0,Corporate,4.0,Credit Card,return risk
4,O1005,C018,Maha Zadeh,maha.zadeh@example.com,italy,Naples,29.01.2026,02/02/2026,Electronics,12,86.96,0.25,782.64,35.0,Home Office,2.0,bank transfer,check address


### 1. Missing Values

In [29]:
missing = pd.DataFrame({
    'missing_count': df.isna().sum(),
}).sort_values('missing_count', ascending=False)

missing
# missing.head(10)

,missing_count
Satisfaction_Score,15
Payment_Method,14
Customer_Segment,12
Email,10
Notes,10
Age,7
Order_Date,5
City,3
Customer_ID,0
Order_ID,0


### 2. Data Types

In [18]:
dtypes_info = pd.DataFrame({
    'dtype':        df.dtypes.astype(str),
    'non_null':     df.notna().sum()
})
dtypes_info

,dtype,non_null
Order_ID,object,100
Customer_ID,object,100
Customer_Name,object,100
Email,object,90
Country,object,100
City,object,97
Order_Date,object,95
Ship_Date,object,100
Product_Category,object,100
Quantity,int64,100


In [19]:
# Flag numeric columns stored as text
for col in df.select_dtypes(include=['object', 'string']).columns:
    sample = df[col].dropna().astype(str).head(50)
    looks_numeric = sample.str.match(r'^-?\d+(\.\d+)?$').mean()
    if looks_numeric > 0.8:
        print(f"  '{col}' looks numeric ({looks_numeric:.0%}) but is stored as text")

  'Unit_Price' looks numeric (88%) but is stored as text


### 3. Value Disctributions

In [20]:
# Numeric columns: Quantity, Unit_Price, Discount, Sales, Age, Satisfaction_Score
df.describe(include=[np.number]).T

,count,mean,std,min,25%,50%,75%,max
Quantity,100.0,5.770000,3.338950,0.0,3.0000,6.000,8.0000,12.00
Discount,100.0,0.130500,0.077491,0.0,0.0500,0.150,0.2000,0.25
Sales,100.0,2477.780800,7144.014096,0.0,376.5775,1246.685,2012.2375,56999.94
Age,93.0,48.494624,19.671003,16.0,35.0000,46.000,60.0000,130.00
Satisfaction_Score,85.0,3.117647,1.247968,1.0,2.0000,3.000,4.0000,5.00


In [21]:
# Categorical / text columns
df.describe(include=['object', 'string', 'category']).T

,count,unique,top,freq
Order_ID,100,95,O1005,2
Customer_ID,100,46,C019,5
Customer_Name,100,39,Nina Weber,8
Email,90,15,nina.weber@example.com,12
Country,100,15,Deutschland,10
City,97,15,Mannheim,10
Order_Date,95,82,04-22-2026,3
Ship_Date,100,87,not available,2
Product_Category,100,10,Office Supplies,19
Unit_Price,100,93,86.96,2


In [22]:
# Top values for low-to-mid cardinality columns
for col in ['Country', 'City', 'Product_Category', 'Customer_Segment', 'Payment_Method']:
    if col in df.columns:
        print(f"\n--- {col} ---")
        print(df[col].value_counts(dropna=False).head(15))


--- Country ---
Country
Deutschland    10
Germany        10
Italy           9
Spain           9
ITALY           8
Netherlands     8
france          7
NETHERLANDS     6
GERMANY         6
italy           6
FRANCE          6
spain           5
germany         4
France          4
España          2
Name: count, dtype: int64

--- City ---
City
Mannheim      10
Karlsruhe     10
Madrid         9
Rome           9
Milan          8
Amsterdam      7
Lyon           6
Naples         6
Heidelberg     6
paris          6
Rotterdam      5
Barcelona      5
karlsruhe      4
Paris          4
NaN            3
Name: count, dtype: int64

--- Product_Category ---
Product_Category
Office Supplies    19
home               12
furniture          12
electronics        10
Electronics        10
Home               10
ELECTRONICS         8
office supplies     7
Furniture           6
OFFICE SUPPLIES     6
Name: count, dtype: int64

--- Customer_Segment ---
Customer_Segment
CORPORATE      23
Consumer       16
home office

### 4. Duplicates

In [23]:
full_dups = df.duplicated().sum()
print(f'Full-row duplicates: {full_dups:,}  ({full_dups / len(df):.2%})')

for col in KEY_COLS:
    if col in df.columns:
        n = df.duplicated(subset=[col]).sum()
        print(f"Duplicates on '{col}': {n:,}")

Full-row duplicates: 5  (5.00%)
Duplicates on 'Order_ID': 5
Duplicates on 'Customer_ID': 54


In [24]:
# Inspect Order_ID duplicates
df[df.duplicated(subset=['Order_ID'], keep=False)].sort_values('Order_ID')

,Order_ID,Customer_ID,Customer_Name,Email,Country,City,Order_Date,Ship_Date,Product_Category,Quantity,Unit_Price,Discount,Sales,Age,Customer_Segment,Satisfaction_Score,Payment_Method,Notes
4,O1005,C018,Maha Zadeh,maha.zadeh@example.com,italy,Naples,29.01.2026,02/02/2026,Electronics,12,86.96,0.25,782.64,35.0,Home Office,2.0,bank transfer,check address
95,O1005,C018,Maha Zadeh,maha.zadeh@example.com,italy,Naples,29.01.2026,02/02/2026,Electronics,12,86.96,0.25,782.64,35.0,Home Office,2.0,bank transfer,check address
17,O1018,C049,MEHDI KARIMI,mehdi.karimi@example.com,Netherlands,Amsterdam,28/01/2026,03/02/2026,Home,8,382.96,0.15,2604.13,58.0,CORPORATE,3.0,NaN,missing phone
96,O1018,C049,MEHDI KARIMI,mehdi.karimi@example.com,Netherlands,Amsterdam,28/01/2026,03/02/2026,Home,8,382.96,0.15,2604.13,58.0,CORPORATE,3.0,NaN,missing phone
28,O1029,C010,NINA WEBER,nina.weber@example.com,NETHERLANDS,Rotterdam,2026-02-06,07/02/2026,Office Supplies,10,€215.16,0.15,1828.86,76.0,Home Office,4.0,Bank Transfer,return risk
97,O1029,C010,NINA WEBER,nina.weber@example.com,NETHERLANDS,Rotterdam,2026-02-06,07/02/2026,Office Supplies,10,€215.16,0.15,1828.86,76.0,Home Office,4.0,Bank Transfer,return risk
47,O1048,C046,Carlos Garcia,carlos.garcia@example.com,Deutschland,Mannheim,01.04.2026,2026-04-02,electronics,2,261.44,0.15,444.45,130.0,CORPORATE,2.0,credit card,urgent
98,O1048,C046,Carlos Garcia,carlos.garcia@example.com,Deutschland,Mannheim,01.04.2026,2026-04-02,electronics,2,261.44,0.15,444.45,130.0,CORPORATE,2.0,credit card,urgent
63,O1064,C034,Nina Weber,nina.weber@example.com,GERMANY,Heidelberg,11/02/2026,02-16-2026,home,5,288.7,0.05,1371.33,56.0,home office,5.0,bank transfer,ok
99,O1064,C034,Nina Weber,nina.weber@example.com,GERMANY,Heidelberg,11/02/2026,02-16-2026,home,5,288.7,0.05,1371.33,56.0,home office,5.0,bank transfer,ok


### 5. Cardinality

In [25]:
cardinality = pd.DataFrame({
    'n_unique':   df.nunique(dropna=True),
    'unique_pct': (df.nunique(dropna=True) / len(df) * 100).round(2),
}).sort_values('n_unique', ascending=False)

cardinality['hint'] = cardinality['unique_pct'].apply(
    lambda p: 'likely ID' if p > 95 else ('low-cardinality categorical' if p < 5 else '')
)
cardinality

,n_unique,unique_pct,hint
Order_ID,95,95.0,
Unit_Price,93,93.0,
Sales,92,92.0,
Ship_Date,87,87.0,
Order_Date,82,82.0,
Customer_ID,46,46.0,
Age,43,43.0,
Customer_Name,39,39.0,
City,15,15.0,
Email,15,15.0,


### 6. Inconsistent text formats

In [26]:
email_re = re.compile(r'^[^@\s]+@[^@\s]+\.[^@\s]+$')
text_issues = {}

for col in df.select_dtypes(include=['object', 'string']).columns:
    s = df[col].dropna().astype(str)
    if s.empty:
        continue

    issues = {
        'leading_trailing_ws': int((s != s.str.strip()).sum()),
        'internal_double_ws':  int(s.str.contains(r'\s{2,}').sum()),
        'mixed_case_variants': int(s.nunique() - s.str.lower().nunique()),
        'empty_strings':       int((s.str.strip() == '').sum()),
        'non_ascii':           int(s.str.contains(r'[^\x00-\x7f]').sum()),
    }

    if col == 'Email':
        issues['invalid_email_format'] = int((~s.str.match(email_re)).sum())

    flagged = {k: v for k, v in issues.items() if v}
    if flagged:
        text_issues[col] = flagged

for col, iss in text_issues.items():
    print(f"  '{col}': {iss}")

  'Customer_Name': {'leading_trailing_ws': 4, 'internal_double_ws': 4, 'mixed_case_variants': 20, 'non_ascii': 11}
  'Country': {'mixed_case_variants': 8, 'non_ascii': 2}
  'City': {'mixed_case_variants': 2}
  'Product_Category': {'mixed_case_variants': 6}
  'Unit_Price': {'non_ascii': 8}
  'Customer_Segment': {'mixed_case_variants': 3}
  'Payment_Method': {'mixed_case_variants': 3}


In [27]:
# Spot variants like 'US' / 'us' / 'USA' or 'Credit Card' / 'credit card'
for col in ['Country', 'City', 'Product_Category', 'Customer_Segment', 'Payment_Method']:
    if col not in df.columns:
        continue
    s = df[col].dropna().astype(str)
    grouped = s.groupby(s.str.lower().str.strip()).unique()
    variants = grouped[grouped.apply(len) > 1]
    if not variants.empty:
        print(f"\n--- {col} — values that collapse to the same key ---")
        for key, vals in variants.items():
            print(f"  '{key}': {list(vals)}")


--- Country — values that collapse to the same key ---
  'france': ['FRANCE', 'France', 'france']
  'germany': ['germany', 'Germany', 'GERMANY']
  'italy': ['Italy', 'italy', 'ITALY']
  'netherlands': ['Netherlands', 'NETHERLANDS']
  'spain': ['Spain', 'spain']

--- City — values that collapse to the same key ---
  'karlsruhe': ['karlsruhe', 'Karlsruhe']
  'paris': ['Paris', 'paris']

--- Product_Category — values that collapse to the same key ---
  'electronics': ['electronics', 'Electronics', 'ELECTRONICS']
  'furniture': ['Furniture', 'furniture']
  'home': ['Home', 'home']
  'office supplies': ['office supplies', 'Office Supplies', 'OFFICE SUPPLIES']

--- Customer_Segment — values that collapse to the same key ---
  'consumer': ['consumer', 'Consumer']
  'corporate': ['Corporate', 'CORPORATE']
  'home office': ['Home Office', 'home office']

--- Payment_Method — values that collapse to the same key ---
  'bank transfer': ['Bank Transfer', 'bank transfer']
  'credit card': ['Credit

### Date Column Checks

In [28]:
for col in DATE_COLS:
    parsed = pd.to_datetime(df[col], errors='coerce')
    unparseable = parsed.isna() & df[col].notna()
    print(f"\n--- {col} ---")
    print(f"  Unparseable values: {unparseable.sum()}")
    print(f"  Min: {parsed.min()}   Max: {parsed.max()}")
    print(f"  Future dates:       {(parsed > pd.Timestamp.today()).sum()}")

# Cross-column: Ship_Date should be on/after Order_Date
order = pd.to_datetime(df['Order_Date'], errors='coerce')
ship  = pd.to_datetime(df['Ship_Date'],  errors='coerce')
print(f"\nShip_Date before Order_Date: {(ship < order).sum()}")
print(f"Ship lag (days) — describe:")
print((ship - order).dt.days.describe())


--- Order_Date ---
  Unparseable values: 81
  Min: 2026-01-15 00:00:00   Max: 2026-05-21 00:00:00
  Future dates:       4

--- Ship_Date ---
  Unparseable values: 2
  Min: 2026-01-10 00:00:00   Max: 2026-11-03 00:00:00
  Future dates:       17

Ship_Date before Order_Date: 0
Ship lag (days) — describe:
count    13.000000
mean      4.461538
std       2.696151
min       1.000000
25%       3.000000
50%       4.000000
75%       6.000000
max      10.000000
dtype: float64


/tmp/ipykernel_6402/1488073578.py:2: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  parsed = pd.to_datetime(df[col], errors='coerce')
/tmp/ipykernel_6402/1488073578.py:11: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  ship  = pd.to_datetime(df['Ship_Date'],  errors='coerce')
